# 7.5 — Transposed & Dilated Convolutions

Dilated convolutions let a small kernel see a wider neighborhood by spacing its weights apart, while transposed convolutions let a coarse feature map learn how to paint a larger grid. In this lesson, we build both operations from scratch with NumPy so the shape formulas, zero insertion, receptive fields, and artifact pitfalls are visible instead of hidden inside a deep-learning library.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build dilation and transposed convolution one idea at a time. Run each cell in order and read the printed intermediate values — every shape, index, and overlap count is exposed so the operations feel like arithmetic rather than magic. This walkthrough is self-contained (it imports what it needs) and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, indexing, and small from-scratch convolution loops.
import matplotlib.pyplot as plt  # visual checks for receptive fields and upsampling artifacts.
np.random.seed(0)  # reproducibility for any random visual examples.

### 1. Dilation widens a kernel without adding weights

A normal 1-D convolution with a 3-tap kernel touches three adjacent input positions. A dilated convolution keeps the same three learned weights but spaces them apart by a dilation rate `d`. The effective footprint is

$$k_{eff}=k+(k-1)(d-1).$$

For `k=3` and `d=2`, the three weights land at offsets `0, 2, 4`, so the filter sees across five input positions while still owning only three parameters.

In [ ]:
k_w = 3  # number of learned weights in the kernel.
d_w = 2  # spacing between neighboring learned taps.
k_eff_w = k_w + (k_w - 1) * (d_w - 1)  # effective footprint width.
tap_offsets_w = np.arange(k_w) * d_w  # actual input offsets touched by the learned weights.
print("effective kernel size:", k_eff_w)
print("tap offsets inside the footprint:", tap_offsets_w)
assert k_eff_w == 5

▶ What you'll see: a 3-weight kernel expands to a width-5 footprint, with taps at offsets 0, 2, and 4.

In [ ]:
footprint_w = np.zeros(k_eff_w)  # mark the full footprint with zeros where no learned weight exists.
footprint_w[tap_offsets_w] = 1  # mark learned tap locations.
plt.figure(figsize=(5, 1.8))
plt.stem(np.arange(k_eff_w), footprint_w, basefmt=" ")
plt.xticks(np.arange(k_eff_w))
plt.yticks([0, 1], ["gap", "tap"])
plt.title("1: dilation=2 spreads 3 taps over width 5")
plt.xlabel("offset inside receptive footprint")
plt.show()

▶ What you'll see: three stems separated by gaps; the gaps are positions inside the receptive field that this kernel does not multiply.

*Why it's done this way:* the formula starts with the first tap, then adds `k-1` jumps of length `d`. A dense kernel would need five learned weights to cover width 5, but dilation spends only three weights and uses the spacing to buy context. That is why dilation is attractive for segmentation: more surrounding evidence without immediately increasing parameter count.

### 2. A dilated dot product skips input values

Dilation changes which input positions contribute to one output. With input `[1, 2, 3, 4, 5]`, weights `[1, 0, -1]`, and dilation `2`, the dot product samples positions `0, 2, 4`: `1·1 + 3·0 + 5·(-1) = -4`. The values `2` and `4` are inside the footprint, but they are skipped by this particular filter.

In [ ]:
x_w = np.array([1., 2., 3., 4., 5.])  # one input signal.
weights_w = np.array([1., 0., -1.])  # a simple edge-like kernel.
d_w = 2  # sample every second position inside the footprint.
idx_w = np.arange(len(weights_w)) * d_w  # input positions used at output index 0.
sampled_w = x_w[idx_w]  # values touched by the dilated kernel.
print("sampled positions:", idx_w)
print("sampled values:", sampled_w)

▶ What you'll see: only input positions 0, 2, and 4 are sampled.

In [ ]:
products_w = sampled_w * weights_w  # per-tap contributions.
y0_w = float(products_w.sum())  # dilated convolution output for this window.
print("per-tap products:", products_w)
print("dilated dot product:", y0_w)
assert y0_w == -4.0

▶ What you'll see: the products are `[1, 0, -5]`, summing to `-4`.

In [ ]:
plt.figure(figsize=(5, 2.6))
plt.bar(np.arange(len(x_w)), x_w, color="lightgray", label="input")
plt.scatter(idx_w, sampled_w, color="crimson", s=90, label="touched")
plt.title("2: a dilated dot product touches interleaved positions")
plt.xlabel("input index")
plt.ylabel("value")
plt.legend()
plt.show()

▶ What you'll see: red markers sit on every other input value; the gray bars between them are skipped.

*Why it's done this way:* convolution is still just multiply-and-sum, but dilation replaces adjacent indexing `i+j` with spaced indexing `i+j·d`. That widens the receptive field, yet it also creates a gridding risk: fine alternating evidence can repeatedly fall into the skipped positions.

### 3. Dilated convolution across a signal

To compute a full valid dilated convolution, slide the same spaced footprint along the input. The number of valid outputs is `n - k_eff + 1`, because the last window must still fit the entire effective footprint inside the signal.

In [ ]:
x_full_w = np.arange(1, 9, dtype=float)  # signal [1,2,...,8].
w_full_w = np.array([1., 0., -1.])  # reuse the edge-like three-tap kernel.
d_full_w = 2  # same dilation rate.
k_eff_full_w = len(w_full_w) + (len(w_full_w) - 1) * (d_full_w - 1)  # width 5.
out_len_w = len(x_full_w) - k_eff_full_w + 1  # valid-output length.
print("input length:", len(x_full_w), "effective kernel:", k_eff_full_w, "output length:", out_len_w)
assert out_len_w == 4

▶ What you'll see: an 8-value signal produces four valid dilated-convolution outputs.

In [ ]:
y_dil_w = []  # collect outputs one window at a time.
for start_w in range(out_len_w):
    positions_w = start_w + np.arange(len(w_full_w)) * d_full_w  # spaced positions for this output.
    y_dil_w.append(float(np.sum(x_full_w[positions_w] * w_full_w)))  # multiply and sum.
y_dil_w = np.array(y_dil_w)
print("dilated conv outputs:", y_dil_w)
assert np.allclose(y_dil_w, [-4., -4., -4., -4.])

▶ What you'll see: every window gives `-4` because this steadily increasing signal has the same two-step difference everywhere.

In [ ]:
plt.figure(figsize=(5, 2.8))
plt.plot(x_full_w, marker="o", label="input")
plt.plot(np.arange(out_len_w) + 2, y_dil_w, marker="s", label="dilated output")
plt.title("3: sliding a dilated footprint")
plt.xlabel("position")
plt.legend()
plt.show()

▶ What you'll see: the output is shorter and flat because each dilated window sees the same slope.

*Why it's done this way:* `k_eff` is the true spatial footprint for shape arithmetic, even though the number of weights is still `k`. The output length is controlled by what must fit on the input grid, so dilation reduces valid length unless padding is added.

### 4. Transposed convolution expands a coarse grid

A transposed convolution with input length `n_in`, stride `s`, padding `p`, kernel size `k`, and output padding `o` has output length

$$n_{out}^{trans}=(n_{in}-1)s-2p+k+o.$$

For `n_in=3`, `s=2`, `p=0`, `k=3`, `o=0`, the output length is `7`. This operation is often called learned upsampling: each coarse input activation writes a shifted copy of the kernel onto a larger output canvas.

In [ ]:
n_in_w, stride_w, pad_w, k_trans_w, out_pad_w = 3, 2, 0, 3, 0  # transposed-conv shape settings.
n_out_w = (n_in_w - 1) * stride_w - 2 * pad_w + k_trans_w + out_pad_w  # output-size formula.
print("transposed output length:", n_out_w)
assert n_out_w == 7

▶ What you'll see: three coarse positions expand to a length-7 output grid.

In [ ]:
coarse_w = np.array([2., 0., 1.])  # a small feature map after downsampling.
kernel_t_w = np.array([1., 2., 1.])  # learned painting stencil.
y_trans_w = np.zeros(n_out_w)  # output canvas.
for i_w, val_w in enumerate(coarse_w):
    start_w = i_w * stride_w - pad_w  # where this input position writes.
    for j_w, wt_w in enumerate(kernel_t_w):
        out_i_w = start_w + j_w
        if 0 <= out_i_w < n_out_w:
            y_trans_w[out_i_w] += val_w * wt_w  # add this activation's shifted stencil.
print("transposed-conv output:", y_trans_w)
assert np.allclose(y_trans_w, [2., 4., 2., 0., 1., 2., 1.])

▶ What you'll see: activation `2` paints `[2,4,2]` at the left; activation `1` paints `[1,2,1]` at the right.

In [ ]:
plt.figure(figsize=(6, 2.7))
plt.stem(np.arange(n_out_w), y_trans_w, basefmt=" ")
plt.title("4: coarse activations paint a larger output")
plt.xlabel("output index")
plt.ylabel("value")
plt.show()

▶ What you'll see: the output is longer than the input, with values placed by shifted learned kernels.

*Why it's done this way:* transposed convolution is not value-inversion of a previous convolution. It is the matrix-transpose shape operation: instead of gathering many input positions into one output, each input position scatters learned contributions into several output positions.

### 5. Output padding chooses between plausible shapes

Stride arithmetic can leave two possible output sizes that differ by one. Output padding resolves that ambiguity in the shape formula, but it does not add new learned weights or new evidence. With `n_in=3`, `s=2`, `p=1`, `k=3`, output padding `0` gives length `5`; output padding `1` gives length `6`.

In [ ]:
n_in_pad_w, s_pad_w, p_pad_w, k_pad_w = 3, 2, 1, 3  # fixed decoder settings.
lengths_pad_w = []  # store sizes for output_padding 0 and 1.
for o_pad_w in [0, 1]:
    length_w = (n_in_pad_w - 1) * s_pad_w - 2 * p_pad_w + k_pad_w + o_pad_w
    lengths_pad_w.append(length_w)
    print("output_padding", o_pad_w, "-> length", length_w)
assert lengths_pad_w == [5, 6]

▶ What you'll see: one parameter changes the output length from 5 to 6.

In [ ]:
plt.figure(figsize=(4.5, 2.8))
plt.bar(["o=0", "o=1"], lengths_pad_w, color=["steelblue", "darkorange"])
plt.title("5: output padding changes shape, not weights")
plt.ylabel("output length")
plt.show()

▶ What you'll see: the second bar is exactly one cell taller.

*Why it's done this way:* padding removes boundary cells from the scattered canvas, while output padding adds a final shape adjustment after that arithmetic. The extra cell is a bookkeeping choice needed to align decoder layers or skip connections; it should not be interpreted as an extra learned boundary rule.

### 6. Uneven overlap creates checkerboard artifacts

For transposed convolution, each input activation writes a kernel-sized patch. If stride and kernel size do not tile evenly, some output positions receive more contributions than others. In 1-D with stride `2` and kernel `3`, input position `0` writes to output `0,1,2`, and input position `1` writes to `2,3,4`, so output position `2` receives two contributions while its neighbors receive one.

In [ ]:
n_inputs_overlap_w, stride_overlap_w, kernel_overlap_w = 4, 2, 3  # settings with uneven overlap.
out_overlap_len_w = (n_inputs_overlap_w - 1) * stride_overlap_w + kernel_overlap_w  # no padding.
coverage_w = np.zeros(out_overlap_len_w)  # count how many stencils cover each output position.
for i_w in range(n_inputs_overlap_w):
    coverage_w[i_w * stride_overlap_w:i_w * stride_overlap_w + kernel_overlap_w] += 1
print("coverage counts:", coverage_w.astype(int))
assert np.array_equal(coverage_w.astype(int), [1, 1, 2, 1, 2, 1, 2, 1, 1])

▶ What you'll see: coverage alternates between one and two contributions in the middle.

In [ ]:
plt.figure(figsize=(6, 2.7))
plt.bar(np.arange(out_overlap_len_w), coverage_w, color="crimson")
plt.title("6: uneven transposed-conv overlap")
plt.xlabel("output index")
plt.ylabel("number of contributing inputs")
plt.show()

▶ What you'll see: alternating tall and short bars; in 2-D this becomes checkerboard texture.

*Why it's done this way:* transposed convolution sums overlapping shifted kernels. When some pixels are summed from more kernels than others, even constant input can produce periodic intensity differences. Resize-then-convolve avoids the arithmetic imbalance by making the upsampled grid dense before applying a normal convolution.

## ✍️ Toy Examples

> ✍️ **Toy examples — trace each dilation/decoder mechanic by hand.** Separate from the walkthrough
> above, here is one tiny, fully hand-traceable toy per computational mechanic in this lesson. Each
> toy prints intermediates with real `# ->` values, draws one picture, and checks the result.

### ✍️ Toy 1 · Dilation widens a kernel footprint without adding weights

A dilated kernel keeps the same learned taps but inserts gaps between where those taps land.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t1_rng = np.random.default_rng(0)                 # seeded generator for this toy
t1_k = 3                                         # -> three learned weights
t1_d = 3                                         # -> spacing of three input steps
t1_k_eff = t1_k + (t1_k - 1) * (t1_d - 1)        # -> 7
print("effective kernel size:", t1_k_eff)        # -> 7
t1_offsets = np.arange(t1_k) * t1_d              # -> [0, 3, 6]
print("tap offsets:", t1_offsets.tolist())       # -> [0, 3, 6]
t1_footprint = np.zeros(t1_k_eff)                # -> length-7 footprint
t1_footprint[t1_offsets] = 1                     # -> [1,0,0,1,0,0,1]
print("tap footprint:", t1_footprint.astype(int).tolist()) # -> [1, 0, 0, 1, 0, 0, 1]
assert t1_k_eff == 7 and np.array_equal(t1_offsets, np.array([0, 3, 6]))

plt.figure(figsize=(5, 1.9))
plt.stem(np.arange(t1_k_eff), t1_footprint, basefmt=" ")
plt.xticks(np.arange(t1_k_eff))
plt.yticks([0, 1], ["gap", "tap"])
plt.title("Toy 1 · dilation spreads taps")
plt.show()

▶ What you'll see: three learned taps span a seven-position footprint.

### ✍️ Toy 2 · A dilated dot product skips values inside the footprint

Dilation changes the indices sampled by one output cell; skipped positions are inside the footprint but
not multiplied by weights.

In [ ]:
import numpy as np

t2_rng = np.random.default_rng(0)                 # seeded generator for this toy
t2_x = np.arange(1, 8, dtype=float)              # -> [1,2,3,4,5,6,7]
print("input signal:", t2_x.tolist())            # -> [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]
t2_w = np.array([2., 0., -1.])                  # -> three learned taps
print("weights:", t2_w.tolist())                # -> [2.0, 0.0, -1.0]
t2_d = 3                                         # -> sample every third input
t2_idx = np.arange(len(t2_w)) * t2_d             # -> [0, 3, 6]
print("sampled indices:", t2_idx.tolist())       # -> [0, 3, 6]
t2_sampled = t2_x[t2_idx]                        # -> [1,4,7]
print("sampled values:", t2_sampled.tolist())    # -> [1.0, 4.0, 7.0]
t2_products = t2_sampled * t2_w                  # -> [2,0,-7]
print("products:", t2_products.tolist())         # -> [2.0, 0.0, -7.0]
t2_y = float(t2_products.sum())                  # -> -5.0
print("dilated dot product:", t2_y)              # -> -5.0
assert t2_y == -5.0

plt.figure(figsize=(5, 2.4))
plt.bar(np.arange(len(t2_x)), t2_x, color="lightgray", label="input")
plt.scatter(t2_idx, t2_sampled, color="crimson", s=90, label="touched")
plt.xlabel("input index")
plt.ylabel("value")
plt.legend()
plt.title("Toy 2 · gaps are skipped")
plt.show()

▶ What you'll see: red markers land on indices `0`, `3`, and `6`; the other values are skipped.

### ✍️ Toy 3 · Valid dilated convolution slides a spaced footprint

The effective kernel size controls how many valid start positions fit inside the signal.

In [ ]:
import numpy as np

t3_rng = np.random.default_rng(0)                 # seeded generator for this toy
t3_x = np.arange(1, 10, dtype=float)             # -> [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0]
print("input signal:", t3_x.tolist())            # -> [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0]
t3_w = np.array([1., 0., -1.])                  # edge-like taps
t3_d = 2                                         # -> spacing two
t3_k_eff = len(t3_w) + (len(t3_w) - 1) * (t3_d - 1) # -> 5
print("effective size:", t3_k_eff)               # -> 5
t3_out_len = len(t3_x) - t3_k_eff + 1            # -> 5
print("output length:", t3_out_len)              # -> 5
t3_positions = []                                # -> touched positions per output
t3_y = []                                        # output values
for t3_start in range(t3_out_len):
    t3_pos = t3_start + np.arange(len(t3_w)) * t3_d
    t3_positions.append(t3_pos.tolist())
    t3_y.append(float(np.sum(t3_x[t3_pos] * t3_w)))
t3_y = np.array(t3_y)                            # -> [-4,-4,-4,-4,-4]
print("touched positions:", t3_positions)        # -> [[0, 2, 4], [1, 3, 5], [2, 4, 6], [3, 5, 7], [4, 6, 8]]
print("dilated outputs:", t3_y.tolist())         # -> [-4.0, -4.0, -4.0, -4.0, -4.0]
assert np.array_equal(t3_y, -4 * np.ones(5))

plt.figure(figsize=(5, 2.5))
plt.plot(t3_x, marker="o", label="input")
plt.plot(np.arange(t3_out_len) + 2, t3_y, marker="s", label="dilated output")
plt.legend()
plt.title("Toy 3 · slide the spaced stencil")
plt.show()

▶ What you'll see: every window has the same two-step difference, so every output is `-4`.

### ✍️ Toy 4 · Transposed convolution scatters shifted kernels

Each coarse activation writes a scaled copy of the kernel onto a larger output canvas.

In [ ]:
import numpy as np

t4_rng = np.random.default_rng(0)                 # seeded generator for this toy
t4_coarse = np.array([1., 2., 0.])               # coarse feature map
print("coarse input:", t4_coarse.tolist())       # -> [1.0, 2.0, 0.0]
t4_kernel = np.array([1., 2., 1.])              # -> painting stencil
print("kernel:", t4_kernel.tolist())             # -> [1.0, 2.0, 1.0]
t4_stride = 2                                    # -> scatter starts every two cells
t4_pad = 0                                       # -> no crop
t4_out_pad = 0                                   # -> no extra output cell
t4_n_out = (len(t4_coarse) - 1) * t4_stride - 2 * t4_pad + len(t4_kernel) + t4_out_pad # -> 7
print("output length:", t4_n_out)                # -> 7
t4_y = np.zeros(t4_n_out)                        # output canvas
t4_paints = []                                   # -> contributions by input activation
for t4_i, t4_val in enumerate(t4_coarse):
    t4_start = t4_i * t4_stride - t4_pad
    t4_this = []
    for t4_j, t4_weight in enumerate(t4_kernel):
        t4_out_i = t4_start + t4_j
        if 0 <= t4_out_i < t4_n_out:
            t4_contrib = t4_val * t4_weight
            t4_y[t4_out_i] += t4_contrib
            t4_this.append((t4_out_i, float(t4_contrib)))
    t4_paints.append(t4_this)
print("painted contributions:", t4_paints)       # -> [[(0,1.0),(1,2.0),(2,1.0)], [(2,2.0),(3,4.0),(4,2.0)], [(4,0.0),(5,0.0),(6,0.0)]]
print("transposed output:", t4_y.tolist())       # -> [1.0, 2.0, 3.0, 4.0, 2.0, 0.0, 0.0]
assert np.array_equal(t4_y, np.array([1., 2., 3., 4., 2., 0., 0.]))

plt.figure(figsize=(5.5, 2.4))
plt.stem(np.arange(t4_n_out), t4_y, basefmt=" ")
plt.xlabel("output index")
plt.ylabel("value")
plt.title("Toy 4 · coarse values paint a canvas")
plt.show()

▶ What you'll see: overlapping writes add at index `2`, producing value `3`.

### ✍️ Toy 5 · Output padding changes transposed-conv shape only

Output padding picks between plausible output lengths after stride and padding arithmetic.

In [ ]:
import numpy as np

t5_rng = np.random.default_rng(0)                 # seeded generator for this toy
t5_n_in = 4                                      # coarse length
t5_stride = 2                                    # -> decoder stride
t5_pad = 1                                       # -> crop one cell per side
t5_kernel = 3                                    # -> kernel length
t5_lengths = []                                  # -> lengths for output_padding 0 and 1
for t5_o in [0, 1]:
    t5_len = (t5_n_in - 1) * t5_stride - 2 * t5_pad + t5_kernel + t5_o
    t5_lengths.append(t5_len)
    print("output_padding", t5_o, "length", t5_len) # -> 0 length 7; 1 length 8
print("length choices:", t5_lengths)             # -> [7, 8]
assert t5_lengths == [7, 8]

plt.figure(figsize=(4.4, 2.5))
plt.bar(["o=0", "o=1"], t5_lengths, color=["steelblue", "darkorange"])
plt.ylabel("output length")
plt.title("Toy 5 · output padding adds one cell")
plt.show()

▶ What you'll see: changing output padding from `0` to `1` changes length from `7` to `8`.

### ✍️ Toy 6 · Uneven transposed-conv overlap makes checkerboard coverage

When stride and kernel size do not tile evenly, some output cells receive more painted stencils than
others. In 2-D, the uneven 1-D pattern multiplies into checkerboard-like coverage.

In [ ]:
import numpy as np

t6_rng = np.random.default_rng(0)                 # seeded generator for this toy
t6_n = 3                                         # -> three coarse positions per axis
t6_stride = 2                                    # -> starts every two cells
t6_kernel = 3                                    # -> each stencil covers three cells
t6_out = (t6_n - 1) * t6_stride + t6_kernel      # -> 7
print("output size per axis:", t6_out)           # -> 7
t6_one = np.zeros(t6_out)                        # -> 1-D coverage counts
for t6_i in range(t6_n):
    t6_one[t6_i*t6_stride:t6_i*t6_stride+t6_kernel] += 1
print("1-D coverage:", t6_one.astype(int).tolist()) # -> [1, 1, 2, 1, 2, 1, 1]
t6_coverage = np.outer(t6_one, t6_one)           # -> 2-D coverage by separable counting
print("2-D coverage shape:", t6_coverage.shape)  # -> (7, 7)
print("center coverage row:", t6_coverage[2].astype(int).tolist()) # -> [2, 2, 4, 2, 4, 2, 2]
print("unique coverage counts:", np.unique(t6_coverage).astype(int).tolist()) # -> [1, 2, 4]
assert t6_coverage.shape == (7, 7) and np.array_equal(np.unique(t6_coverage).astype(int), [1, 2, 4])

plt.figure(figsize=(4.2, 3.4))
plt.imshow(t6_coverage, cmap="magma")
plt.colorbar(label="number of writes")
plt.title("Toy 6 · uneven 2-D overlap")
plt.show()

▶ What you'll see: alternating cells receive 1, 2, or 4 writes — the source of checkerboard texture.

### ✍️ Toy 7 · Resize-then-convolve fills a dense grid before smoothing

Nearest-neighbor resize creates a dense upsampled signal first; a normal convolution then smooths that
signal without zero-insertion gaps.

In [ ]:
import numpy as np

t7_rng = np.random.default_rng(0)                 # seeded generator for this toy
t7_coarse = np.array([1., 2., 1.])               # coarse signal
print("coarse signal:", t7_coarse.tolist())      # -> [1.0, 2.0, 1.0]
t7_upsampled = np.repeat(t7_coarse, 2)           # -> [1,1,2,2,1,1]
print("nearest upsampled:", t7_upsampled.tolist()) # -> [1.0, 1.0, 2.0, 2.0, 1.0, 1.0]
t7_kernel = np.array([0.25, 0.5, 0.25])          # -> smoothing kernel
print("smoothing kernel:", t7_kernel.tolist())   # -> [0.25, 0.5, 0.25]
t7_padded = np.pad(t7_upsampled, 1, mode="edge") # edge-padded dense signal
print("edge-padded signal:", t7_padded.tolist()) # -> [1.0, 1.0, 1.0, 2.0, 2.0, 1.0, 1.0, 1.0]
t7_smooth = []                                   # -> smoothed output
for t7_i in range(len(t7_upsampled)):
    t7_smooth.append(float(np.sum(t7_padded[t7_i:t7_i+3] * t7_kernel)))
t7_smooth = np.array(t7_smooth)                  # -> [1,1.25,1.75,1.75,1.25,1]
print("resize-then-convolve output:", t7_smooth.tolist()) # -> [1.0, 1.25, 1.75, 1.75, 1.25, 1.0]
assert np.allclose(t7_smooth, [1.0, 1.25, 1.75, 1.75, 1.25, 1.0])

plt.figure(figsize=(5.2, 2.5))
plt.plot(t7_upsampled, marker="o", label="nearest")
plt.plot(t7_smooth, marker="s", label="smoothed")
plt.legend()
plt.title("Toy 7 · dense resize before convolution")
plt.show()

▶ What you'll see: the upsampled signal is dense first, then smoothing makes a gradual transition.


## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for arrays, indexing, dot products, and from-scratch convolution loops.
import matplotlib.pyplot as plt # load Matplotlib for heatmaps, stems, bars, and artifact diagnostics.
np.random.seed(0) # make every random example reproducible.

## 🟢 Basics (warm-up)

### Basic 1 — Compute effective kernel size

**Goal.** Compute the effective width of a dilated kernel, because the footprint controls which inputs an output can see. We build it in 2 steps.

In [ ]:
k_b1 = 3 # count learned kernel taps.
d_b1 = 2 # set spacing between adjacent taps.
k_eff_b1 = k_b1 + (k_b1 - 1) * (d_b1 - 1) # compute effective footprint width.
print("effective size:", k_eff_b1) # inspect the widened footprint.
assert k_eff_b1 == 5 # verify the canonical lesson number.

▶ What you'll see: the 3-tap kernel has a width-5 receptive footprint.

In [ ]:
taps_b1 = np.arange(k_b1) * d_b1 # locate learned taps inside that footprint.
mask_b1 = np.zeros(k_eff_b1) # create a footprint mask.
mask_b1[taps_b1] = 1 # mark the learned tap positions.
print("tap offsets:", taps_b1) # inspect positions 0, 2, 4.
plt.figure(figsize=(5, 1.8)) # create a compact footprint plot.
plt.stem(np.arange(k_eff_b1), mask_b1, basefmt=" ") # show taps and gaps.
plt.title("Basic 1: dilated footprint") # title the plot.
plt.xlabel("offset") # label the footprint axis.
plt.show() # display the plot.

▶ What you'll see: the tap mask has gaps between learned weights.

👀 Takeaway: dilation widens the receptive field without increasing the number of learned weights.

### Basic 2 — Place dilated taps on an input

**Goal.** Identify the exact input positions touched by one dilated output, because dilation is easiest to understand as index arithmetic. We build it in 2 steps.

In [ ]:
x_b2 = np.array([10., 20., 30., 40., 50.]) # define a five-value input.
k_b2 = 3 # use three learned taps.
d_b2 = 2 # skip one input between taps.
pos_b2 = np.arange(k_b2) * d_b2 # compute touched positions for the first output.
print("positions touched:", pos_b2) # inspect spaced positions.
print("values touched:", x_b2[pos_b2]) # inspect sampled values.

▶ What you'll see: positions 0, 2, and 4 are touched; positions 1 and 3 are inside the footprint but skipped.

In [ ]:
plt.figure(figsize=(5, 2.5)) # create a bar plot for the sampled input.
plt.bar(np.arange(len(x_b2)), x_b2, color="lightgray") # draw all input values.
plt.scatter(pos_b2, x_b2[pos_b2], color="crimson", s=90) # highlight touched values.
plt.title("Basic 2: touched vs skipped inputs") # title the diagnostic.
plt.xlabel("input index") # label input positions.
plt.ylabel("value") # label values.
plt.show() # display the plot.

▶ What you'll see: highlighted points land on alternating input positions.

👀 Takeaway: dilation changes the sampling pattern, not the multiply-and-sum rule.

### Basic 3 — Do one dilated dot product

**Goal.** Compute one dilated convolution output from sampled values and weights, because every larger example repeats this same dot product. We build it in 2 steps.

In [ ]:
x_b3 = np.array([1., 2., 3., 4., 5.]) # define the input window.
w_b3 = np.array([1., 0., -1.]) # define the three learned weights.
d_b3 = 2 # choose dilation 2.
sampled_b3 = x_b3[np.arange(len(w_b3)) * d_b3] # gather spaced input values.
print("sampled values:", sampled_b3) # inspect the values used in the dot product.

▶ What you'll see: the dot product will use `[1, 3, 5]`.

In [ ]:
products_b3 = sampled_b3 * w_b3 # multiply sampled inputs by learned weights.
y_b3 = float(products_b3.sum()) # sum products into one output.
print("products:", products_b3, "output:", y_b3) # inspect the arithmetic.
assert y_b3 == -4.0 # verify 1*1 + 3*0 + 5*(-1).
plt.figure(figsize=(4, 2.6)) # create a contribution plot.
plt.bar(["tap0", "tap1", "tap2"], products_b3, color="teal") # show per-tap products.
plt.title("Basic 3: dilated dot-product contributions") # title the plot.
plt.show() # display the plot.

▶ What you'll see: the negative last product dominates the sum.

👀 Takeaway: a dilated convolution output is still a dot product over selected input positions.

### Basic 4 — Compare dense and dilated footprints

**Goal.** Compare `d=1` and `d=2`, because changing only dilation changes spatial reach while keeping parameter count fixed. We build it in 2 steps.

In [ ]:
k_b4 = 3 # keep three learned weights in both cases.
dense_offsets_b4 = np.arange(k_b4) * 1 # ordinary convolution offsets.
dilated_offsets_b4 = np.arange(k_b4) * 2 # dilation-2 offsets.
print("dense offsets:", dense_offsets_b4) # inspect adjacent taps.
print("dilated offsets:", dilated_offsets_b4) # inspect spaced taps.

▶ What you'll see: dense taps are `[0,1,2]`; dilated taps are `[0,2,4]`.

In [ ]:
plt.figure(figsize=(5, 2.4)) # create a footprint comparison.
plt.scatter(dense_offsets_b4, np.zeros_like(dense_offsets_b4), s=90, label="d=1") # plot dense taps.
plt.scatter(dilated_offsets_b4, np.ones_like(dilated_offsets_b4), s=90, label="d=2") # plot dilated taps.
plt.yticks([0, 1], ["dense", "dilated"]) # label rows.
plt.xticks(np.arange(5)) # show all possible offsets.
plt.title("Basic 4: same 3 weights, wider reach") # title the plot.
plt.legend() # show labels.
plt.show() # display the plot.

▶ What you'll see: both rows have three points, but the dilated row spans a wider interval.

👀 Takeaway: dilation buys receptive-field width by inserting gaps between taps.

### Basic 5 — Compute transposed-convolution output size

**Goal.** Use the transposed-convolution shape formula, because decoder layers must hit exact spatial sizes. We build it in 2 steps.

In [ ]:
n_in_b5 = 3 # input length.
s_b5 = 2 # stride.
p_b5 = 0 # padding.
k_b5 = 3 # kernel size.
o_b5 = 0 # output padding.
n_out_b5 = (n_in_b5 - 1) * s_b5 - 2 * p_b5 + k_b5 + o_b5 # transposed output length.
print("output length:", n_out_b5) # inspect the result.
assert n_out_b5 == 7 # verify the lesson calculation.

▶ What you'll see: length 3 expands to length 7.

In [ ]:
plt.figure(figsize=(4, 2.5)) # create a simple shape bar chart.
plt.bar(["input", "output"], [n_in_b5, n_out_b5], color=["gray", "seagreen"]) # compare lengths.
plt.title("Basic 5: transposed-conv shape expansion") # title the plot.
plt.ylabel("length") # label size axis.
plt.show() # display the plot.

▶ What you'll see: the output bar is much longer than the input bar.

👀 Takeaway: transposed convolution is a learned shape-expansion operation.

### Basic 6 — Insert zeros between coarse activations

**Goal.** Create the zero-inserted grid behind transposed convolution, because stride expansion spreads input positions apart before filtering. We build it in 2 steps.

In [ ]:
x_b6 = np.array([2., 0., 1.]) # coarse input activations.
s_b6 = 2 # stride means one zero gap between original samples.
expanded_b6 = np.zeros((len(x_b6) - 1) * s_b6 + 1) # allocate expanded grid.
expanded_b6[::s_b6] = x_b6 # place original activations every stride positions.
print("expanded grid:", expanded_b6) # inspect inserted zeros.
assert np.allclose(expanded_b6, [2., 0., 0., 0., 1.])

▶ What you'll see: original values land at indices 0, 2, and 4, with zeros between them.

In [ ]:
plt.figure(figsize=(5, 2.5)) # create a stem plot of the expanded grid.
plt.stem(np.arange(len(expanded_b6)), expanded_b6, basefmt=" ") # show inserted-zero positions.
plt.title("Basic 6: stride-2 zero insertion") # title the plot.
plt.xlabel("expanded index") # label positions.
plt.show() # display the plot.

▶ What you'll see: many grid positions are zero before the learned filter is applied.

👀 Takeaway: transposed convolution inserts gaps between input positions, not inside the kernel footprint.

### Basic 7 — Paint one activation with a kernel

**Goal.** Show how one coarse activation contributes to multiple output positions, because transposed convolution scatters rather than gathers. We build it in 2 steps.

In [ ]:
activation_b7 = 2.0 # one coarse feature value.
kernel_b7 = np.array([1., 2., 1.]) # learned stencil.
paint_b7 = activation_b7 * kernel_b7 # contribution written by this activation.
print("painted contribution:", paint_b7) # inspect activation-scaled weights.
assert np.allclose(paint_b7, [2., 4., 2.])

▶ What you'll see: one scalar becomes a three-position pattern.

In [ ]:
plt.figure(figsize=(4, 2.5)) # create a contribution plot.
plt.bar(np.arange(len(paint_b7)), paint_b7, color="darkorange") # show where one activation writes.
plt.title("Basic 7: one activation paints a stencil") # title the plot.
plt.xlabel("local output offset") # label offsets.
plt.ylabel("contribution") # label contribution scale.
plt.show() # display the plot.

▶ What you'll see: the center output receives the largest contribution because the middle weight is 2.

👀 Takeaway: transposed-conv weights decide how each coarse activation is distributed spatially.

### Basic 8 — Sum overlapping painted stencils

**Goal.** Add contributions from multiple coarse positions, because transposed convolution outputs are sums where stencils overlap. We build it in 2 steps.

In [ ]:
x_b8 = np.array([1., 1.]) # two adjacent coarse activations.
kernel_b8 = np.array([1., 2., 1.]) # a three-tap stencil.
s_b8 = 2 # stride between painted stencils.
y_b8 = np.zeros((len(x_b8) - 1) * s_b8 + len(kernel_b8)) # allocate output canvas.
for i_b8, val_b8 in enumerate(x_b8): # scatter each activation.
    y_b8[i_b8 * s_b8:i_b8 * s_b8 + len(kernel_b8)] += val_b8 * kernel_b8 # add shifted stencil.
print("overlap-summed output:", y_b8) # inspect summed output.
assert np.allclose(y_b8, [1., 2., 2., 2., 1.])

▶ What you'll see: index 2 receives two contributions, so it equals 2.

In [ ]:
plt.figure(figsize=(5, 2.5)) # create a stem plot for the output.
plt.stem(np.arange(len(y_b8)), y_b8, basefmt=" ") # show the summed output.
plt.title("Basic 8: overlapping transposed stencils") # title the plot.
plt.xlabel("output index") # label output positions.
plt.show() # display the plot.

▶ What you'll see: the middle value is raised by overlap from both input activations.

👀 Takeaway: transposed convolution sums all stencil contributions that land on the same output cell.

### Basic 9 — Compare output padding values

**Goal.** See how output padding changes only the target length, because shape alignment often differs by one cell in decoders. We build it in 2 steps.

In [ ]:
n_in_b9, s_b9, p_b9, k_b9 = 3, 2, 1, 3 # fixed transposed-conv settings.
length0_b9 = (n_in_b9 - 1) * s_b9 - 2 * p_b9 + k_b9 + 0 # output_padding 0.
length1_b9 = (n_in_b9 - 1) * s_b9 - 2 * p_b9 + k_b9 + 1 # output_padding 1.
print("o=0 length:", length0_b9, "o=1 length:", length1_b9) # inspect the one-cell difference.
assert (length0_b9, length1_b9) == (5, 6)

▶ What you'll see: the two legal shapes differ by exactly one.

In [ ]:
plt.figure(figsize=(4, 2.5)) # create a length comparison plot.
plt.bar(["output_padding=0", "output_padding=1"], [length0_b9, length1_b9], color=["steelblue", "orange"]) # compare shapes.
plt.title("Basic 9: output padding selects shape") # title the plot.
plt.ylabel("length") # label shape axis.
plt.xticks(rotation=15) # rotate labels.
plt.show() # display the plot.

▶ What you'll see: output padding 1 adds one output cell.

👀 Takeaway: output padding is a shape disambiguation knob, not a new source of learned content.

### Basic 10 — Count uneven overlap

**Goal.** Count how many input stencils cover each output position, because uneven coverage is the arithmetic root of checkerboards. We build it in 2 steps.

In [ ]:
n_b10 = 4 # number of coarse input positions.
s_b10 = 2 # stride between their painted stencils.
k_b10 = 3 # stencil width.
coverage_b10 = np.zeros((n_b10 - 1) * s_b10 + k_b10) # allocate output coverage counts.
for i_b10 in range(n_b10): # visit each coarse position.
    coverage_b10[i_b10 * s_b10:i_b10 * s_b10 + k_b10] += 1 # count one contribution across its stencil.
print("coverage:", coverage_b10.astype(int)) # inspect contribution counts.
assert np.array_equal(coverage_b10.astype(int), [1, 1, 2, 1, 2, 1, 2, 1, 1])

▶ What you'll see: coverage alternates between one and two contributions.

In [ ]:
plt.figure(figsize=(6, 2.5)) # create a coverage chart.
plt.bar(np.arange(len(coverage_b10)), coverage_b10, color="crimson") # visualize uneven overlap.
plt.title("Basic 10: uneven overlap counts") # title the plot.
plt.xlabel("output index") # label output cells.
plt.ylabel("contributors") # label count axis.
plt.show() # display the plot.

▶ What you'll see: alternating bars indicate where checkerboard energy can appear.

👀 Takeaway: stride-kernel pairs with uneven overlap can bake periodic artifacts into the output geometry.

## 🟡 Easy

### Easy 1 — Implement valid dilated convolution

**Goal.** Write a small from-scratch dilated convolution loop, because the library operation is only repeated spaced dot products. We build it in 3 steps.

In [ ]:
x_e1 = np.arange(1, 9, dtype=float) # define input signal [1,...,8].
w_e1 = np.array([1., 0., -1.]) # define an edge-like kernel.
d_e1 = 2 # set dilation rate.
k_eff_e1 = len(w_e1) + (len(w_e1) - 1) * (d_e1 - 1) # compute effective width.
out_len_e1 = len(x_e1) - k_eff_e1 + 1 # compute valid output length.
print("effective width:", k_eff_e1, "output length:", out_len_e1) # inspect shape arithmetic.

▶ What you'll see: width 5 and output length 4.

In [ ]:
y_e1 = np.zeros(out_len_e1) # allocate the output signal.
for i_e1 in range(out_len_e1): # slide the dilated footprint.
    positions_e1 = i_e1 + np.arange(len(w_e1)) * d_e1 # positions touched by this output.
    y_e1[i_e1] = np.sum(x_e1[positions_e1] * w_e1) # spaced dot product.
print("dilated convolution:", y_e1) # inspect outputs.
assert np.allclose(y_e1, [-4., -4., -4., -4.])

In [ ]:
plt.figure(figsize=(5, 2.8)) # create a signal comparison plot.
plt.plot(x_e1, marker="o", label="input") # draw input.
plt.plot(np.arange(out_len_e1) + 2, y_e1, marker="s", label="dilated output") # draw aligned output.
plt.title("Easy 1: valid dilated convolution") # title the plot.
plt.legend() # show curve labels.
plt.show() # display the plot.

▶ What you'll see: a short flat output from repeated two-step differences.

👀 Takeaway: implementing dilation only requires changing the input index stride inside each dot product.

### Easy 2 — Compare dilation rates on the same signal

**Goal.** Run the same kernel with several dilation rates, because dilation is a receptive-field knob. We build it in 3 steps.

In [ ]:
x_e2 = np.array([0., 1., 3., 6., 10., 15., 21., 28.]) # a curved signal with changing differences.
w_e2 = np.array([1., 0., -1.]) # compare left and right sampled values.
dilations_e2 = np.array([1, 2, 3]) # test several spacings.
print("dilations:", dilations_e2) # inspect the sweep.

▶ What you'll see: the same three weights will view increasingly wide neighborhoods.

In [ ]:
outputs_e2 = [] # store one output vector per dilation.
for d_cur_e2 in dilations_e2: # loop over dilation rates.
    k_eff_cur_e2 = len(w_e2) + (len(w_e2) - 1) * (d_cur_e2 - 1) # effective footprint.
    out_cur_e2 = [] # collect outputs for this dilation.
    for i_cur_e2 in range(len(x_e2) - k_eff_cur_e2 + 1): # valid starts.
        pos_cur_e2 = i_cur_e2 + np.arange(len(w_e2)) * d_cur_e2 # sampled positions.
        out_cur_e2.append(float(np.sum(x_e2[pos_cur_e2] * w_e2))) # spaced dot product.
    outputs_e2.append(np.array(out_cur_e2)) # store current output.
print("first outputs:", [float(o_e2[0]) for o_e2 in outputs_e2]) # inspect how wider spacing changes magnitude.
assert [float(o_e2[0]) for o_e2 in outputs_e2] == [-3.0, -10.0, -21.0]

In [ ]:
plt.figure(figsize=(5.5, 3)) # create a dilation-sweep plot.
for d_cur_e2, out_cur_e2 in zip(dilations_e2, outputs_e2): # draw each result.
    plt.plot(out_cur_e2, marker="o", label=f"d={d_cur_e2}") # plot output sequence.
plt.title("Easy 2: dilation changes the measured scale") # title the plot.
plt.xlabel("output index") # label outputs.
plt.ylabel("response") # label response.
plt.legend() # show dilation labels.
plt.show() # display the plot.

▶ What you'll see: larger dilation produces larger negative responses on this increasing curved signal.

👀 Takeaway: dilation can detect broader-scale differences with the same learned kernel.

### Easy 3 — Implement 1-D transposed convolution

**Goal.** Build a transposed convolution by scattering shifted kernels, because that exposes why output cells can overlap. We build it in 3 steps.

In [ ]:
x_e3 = np.array([2., 0., 1.]) # coarse input feature map.
w_e3 = np.array([1., 2., 1.]) # learned upsampling kernel.
s_e3 = 2 # stride between coarse positions.
p_e3 = 0 # no output cropping from padding.
o_e3 = 0 # no output-padding extension.
n_out_e3 = (len(x_e3) - 1) * s_e3 - 2 * p_e3 + len(w_e3) + o_e3 # output length.
y_e3 = np.zeros(n_out_e3) # allocate output canvas.
print("output length:", n_out_e3) # inspect formula result.

▶ What you'll see: the output canvas has length 7.

In [ ]:
for i_e3, val_e3 in enumerate(x_e3): # scatter each input activation.
    for j_e3, wt_e3 in enumerate(w_e3): # loop over learned stencil weights.
        out_i_e3 = i_e3 * s_e3 - p_e3 + j_e3 # destination index.
        if 0 <= out_i_e3 < n_out_e3: # keep only valid output cells.
            y_e3[out_i_e3] += val_e3 * wt_e3 # add the shifted contribution.
print("transposed convolution:", y_e3) # inspect output values.
assert np.allclose(y_e3, [2., 4., 2., 0., 1., 2., 1.])

In [ ]:
plt.figure(figsize=(6, 2.8)) # create a stem plot.
plt.stem(np.arange(n_out_e3), y_e3, basefmt=" ") # draw transposed-conv output.
plt.title("Easy 3: from-scratch transposed convolution") # title the plot.
plt.xlabel("output index") # label output cells.
plt.show() # display the plot.

▶ What you'll see: two nonzero input activations paint two separated `[1,2,1]` patterns.

👀 Takeaway: transposed convolution expands by scattering learned kernels and summing their overlaps.

### Easy 4 — Show output padding in an implementation

**Goal.** Compare transposed-conv outputs with `output_padding=0` and `1`, because the extra cell changes shape without changing the kernel. We build it in 3 steps.

In [ ]:
x_e4 = np.array([1., 2., 1.]) # coarse input.
w_e4 = np.array([1., 0., -1.]) # learned stencil.
s_e4 = 2 # stride.
p_e4 = 1 # crop one cell from each boundary in the formula.
lengths_e4 = [(len(x_e4) - 1) * s_e4 - 2 * p_e4 + len(w_e4) + o_cur_e4 for o_cur_e4 in [0, 1]] # output sizes.
print("lengths:", lengths_e4) # inspect shape choices.
assert lengths_e4 == [5, 6]

▶ What you'll see: the two output-padding settings request lengths 5 and 6.

In [ ]:
outs_e4 = [] # collect outputs for output padding 0 and 1.
for o_cur_e4, n_out_cur_e4 in zip([0, 1], lengths_e4): # run both settings.
    y_cur_e4 = np.zeros(n_out_cur_e4) # allocate output canvas.
    for i_cur_e4, val_cur_e4 in enumerate(x_e4): # scatter each input.
        for j_cur_e4, wt_cur_e4 in enumerate(w_e4): # scatter each stencil tap.
            out_i_cur_e4 = i_cur_e4 * s_e4 - p_e4 + j_cur_e4 # destination after padding crop.
            if 0 <= out_i_cur_e4 < n_out_cur_e4: # only write inside requested output length.
                y_cur_e4[out_i_cur_e4] += val_cur_e4 * wt_cur_e4 # add contribution.
    outs_e4.append(y_cur_e4) # store result.
print("o=0 output:", outs_e4[0]) # inspect first shape.
print("o=1 output:", outs_e4[1]) # inspect second shape.

In [ ]:
plt.figure(figsize=(6, 2.8)) # create a shape comparison plot.
plt.plot(outs_e4[0], marker="o", label="o=0") # draw length-5 output.
plt.plot(outs_e4[1], marker="s", label="o=1") # draw length-6 output.
plt.title("Easy 4: output padding changes requested length") # title the plot.
plt.xlabel("output index") # label output cells.
plt.legend() # show output-padding labels.
plt.show() # display the plot.

▶ What you'll see: the `o=1` curve has one extra index at the right boundary.

👀 Takeaway: output padding is about shape alignment, not adding another learned stencil.

### Easy 5 — Compare transposed convolution with resize-then-convolve

**Goal.** Contrast uneven transposed overlap with a resize-then-convolve baseline, because many decoders use resizing to reduce checkerboard artifacts. We build it in 3 steps.

In [ ]:
x_e5 = np.ones(5) # constant coarse input so artifacts come from geometry, not data.
w_e5 = np.ones(3) # all-one kernel makes coverage directly visible.
s_e5 = 2 # stride-2 upsampling.
y_trans_e5 = np.zeros((len(x_e5) - 1) * s_e5 + len(w_e5)) # transposed-conv canvas.
for i_e5, val_e5 in enumerate(x_e5): # scatter each coarse activation.
    y_trans_e5[i_e5 * s_e5:i_e5 * s_e5 + len(w_e5)] += val_e5 * w_e5 # add all-one stencil.
print("transposed output:", y_trans_e5) # inspect uneven coverage pattern.

▶ What you'll see: the transposed output alternates between lower and higher values.

In [ ]:
upsampled_e5 = np.repeat(x_e5, s_e5) # nearest-neighbor resize to a dense grid.
y_resize_e5 = np.convolve(upsampled_e5, w_e5, mode="same") # ordinary dense convolution after resizing.
print("resize-then-conv output:", y_resize_e5) # inspect the smoother coverage.
assert y_trans_e5.max() == 2.0 and y_resize_e5[2:-2].min() == 3.0

In [ ]:
plt.figure(figsize=(6, 3)) # create a comparison plot.
plt.plot(y_trans_e5, marker="o", label="transposed conv") # draw uneven-overlap output.
plt.plot(y_resize_e5, marker="s", label="resize then conv") # draw dense-grid output.
plt.title("Easy 5: uneven overlap vs dense resize") # title the plot.
plt.xlabel("output index") # label output cells.
plt.ylabel("value") # label response.
plt.legend() # show method labels.
plt.show() # display the plot.

▶ What you'll see: transposed convolution has an alternating pattern; resize-then-convolve has uniform interior coverage.

👀 Takeaway: checkerboards can be reduced by making coverage uniform before learning the convolution.

## 🔴 Advanced

### Advanced 1 — Build a 2-D dilated convolution

**Goal.** Implement a small 2-D dilated convolution, because image dilation spaces kernel taps along both height and width. We build it in 4 steps.

In [ ]:
img_a1 = np.arange(1, 26, dtype=float).reshape(5, 5) # define a 5x5 image with increasing values.
kernel_a1 = np.array([[1., 0.], [0., -1.]]) # define a 2x2 diagonal-difference kernel.
d_a1 = 2 # use dilation 2 in both axes.
k_eff_h_a1 = kernel_a1.shape[0] + (kernel_a1.shape[0] - 1) * (d_a1 - 1) # effective height.
k_eff_w_a1 = kernel_a1.shape[1] + (kernel_a1.shape[1] - 1) * (d_a1 - 1) # effective width.
out_shape_a1 = (img_a1.shape[0] - k_eff_h_a1 + 1, img_a1.shape[1] - k_eff_w_a1 + 1) # valid output shape.
print("effective kernel:", (k_eff_h_a1, k_eff_w_a1), "output shape:", out_shape_a1) # inspect shapes.
assert out_shape_a1 == (3, 3)

▶ What you'll see: a 2×2 kernel at dilation 2 behaves like a 3×3 footprint and produces a 3×3 output.

In [ ]:
y_a1 = np.zeros(out_shape_a1) # allocate valid-convolution output.
for r_a1 in range(out_shape_a1[0]): # slide over output rows.
    for c_a1 in range(out_shape_a1[1]): # slide over output columns.
        total_a1 = 0.0 # accumulate one output value.
        for kr_a1 in range(kernel_a1.shape[0]): # loop over kernel rows.
            for kc_a1 in range(kernel_a1.shape[1]): # loop over kernel columns.
                total_a1 += img_a1[r_a1 + kr_a1 * d_a1, c_a1 + kc_a1 * d_a1] * kernel_a1[kr_a1, kc_a1] # spaced 2-D tap.
        y_a1[r_a1, c_a1] = total_a1 # store output.
print("2-D dilated output:\n", y_a1) # inspect result.
assert np.allclose(y_a1, -12 * np.ones((3, 3)))

In [ ]:
plt.figure(figsize=(4, 3)) # create input heatmap.
plt.imshow(img_a1, cmap="viridis") # visualize the original image.
plt.colorbar(label="input value") # add color scale.
plt.title("Advanced 1: input image") # title input.
plt.show() # display input.

▶ What you'll see: a smooth increasing 5×5 grid.

In [ ]:
plt.figure(figsize=(4, 3)) # create output heatmap.
plt.imshow(y_a1, cmap="coolwarm") # visualize dilated-filter response.
plt.colorbar(label="response") # add color scale.
plt.title("Advanced 1: 2-D dilated response") # title output.
plt.show() # display output.

▶ What you'll see: a constant negative response because every spaced diagonal difference is the same.

👀 Takeaway: 2-D dilation applies the same spacing idea independently across rows and columns.

### Advanced 2 — Visualize gridding from repeated dilation

**Goal.** Show how large dilation can miss interleaved evidence, because repeatedly sampled grids may ignore alternating patterns. We build it in 4 steps.

In [ ]:
pattern_a2 = (np.indices((8, 8)).sum(axis=0) % 2).astype(float) # create a checkerboard image.
kernel_a2 = np.ones((3, 3)) # count sampled ones inside a 3x3 dilated footprint.
d_a2 = 2 # sample every other pixel in both axes.
out_shape_a2 = (pattern_a2.shape[0] - (3 + 2) + 1, pattern_a2.shape[1] - (3 + 2) + 1) # valid shape for k_eff=5.
print("checkerboard ones:", int(pattern_a2.sum()), "output shape:", out_shape_a2) # inspect setup.

▶ What you'll see: half the pixels are ones, but the dilated kernel will sample only one parity per window.

In [ ]:
y_a2 = np.zeros(out_shape_a2) # allocate response map.
for r_a2 in range(out_shape_a2[0]): # output rows.
    for c_a2 in range(out_shape_a2[1]): # output columns.
        total_a2 = 0.0 # accumulate sampled checkerboard values.
        for kr_a2 in range(3): # kernel rows.
            for kc_a2 in range(3): # kernel columns.
                total_a2 += pattern_a2[r_a2 + kr_a2 * d_a2, c_a2 + kc_a2 * d_a2] # spaced sample.
        y_a2[r_a2, c_a2] = total_a2 # store count.
print("sampled-one counts:\n", y_a2) # inspect gridding response.
assert set(np.unique(y_a2)) == {0.0, 9.0}

In [ ]:
fig_a2, ax_a2 = plt.subplots(1, 2, figsize=(7, 3)) # create side-by-side visualization.
ax_a2[0].imshow(pattern_a2, cmap="gray") # show checkerboard input.
ax_a2[0].set_title("input checkerboard") # title input.
ax_a2[1].imshow(y_a2, cmap="magma") # show dilated sample counts.
ax_a2[1].set_title("dilated sampled counts") # title output.
plt.show() # display both plots.

▶ What you'll see: the response flips between all-zero and all-nine windows depending on alignment.

In [ ]:
plt.figure(figsize=(4, 2.8)) # create a histogram.
plt.hist(y_a2.ravel(), bins=[-0.5, 0.5, 8.5, 9.5], color="purple") # count response values.
plt.title("Advanced 2: gridding response histogram") # title histogram.
plt.xlabel("sampled-one count") # label count.
plt.ylabel("frequency") # label frequency.
plt.show() # display histogram.

▶ What you'll see: responses cluster at the extremes, showing how dilation can alias fine patterns.

👀 Takeaway: dilation increases reach, but repeated gaps can create blind spots for interleaved structure.

### Advanced 3 — Build 2-D transposed convolution

**Goal.** Implement 2-D transposed convolution by scattering patches, because image decoders do this across height and width. We build it in 4 steps.

In [ ]:
x_a3 = np.array([[1., 2.], [0., 1.]]) # define a 2x2 coarse feature map.
kernel_a3 = np.array([[1., 0.], [0., 1.]]) # define a 2x2 learned stencil.
s_a3 = 2 # stride in both axes.
out_shape_a3 = ((x_a3.shape[0] - 1) * s_a3 + kernel_a3.shape[0], (x_a3.shape[1] - 1) * s_a3 + kernel_a3.shape[1]) # output shape.
y_a3 = np.zeros(out_shape_a3) # allocate output canvas.
print("output shape:", out_shape_a3) # inspect shape.
assert out_shape_a3 == (4, 4)

▶ What you'll see: a 2×2 coarse map expands to a 4×4 output.

In [ ]:
for r_a3 in range(x_a3.shape[0]): # input rows.
    for c_a3 in range(x_a3.shape[1]): # input columns.
        y_a3[r_a3 * s_a3:r_a3 * s_a3 + 2, c_a3 * s_a3:c_a3 * s_a3 + 2] += x_a3[r_a3, c_a3] * kernel_a3 # scatter patch.
print("2-D transposed output:\n", y_a3) # inspect output grid.
assert y_a3[0, 0] == 1.0 and y_a3[0, 2] == 2.0 and y_a3[3, 3] == 1.0

In [ ]:
fig_a3, ax_a3 = plt.subplots(1, 2, figsize=(7, 3)) # create input-output figure.
ax_a3[0].imshow(x_a3, cmap="viridis") # show coarse input.
ax_a3[0].set_title("coarse input") # title input.
ax_a3[1].imshow(y_a3, cmap="viridis") # show expanded output.
ax_a3[1].set_title("transposed output") # title output.
plt.show() # display both.

▶ What you'll see: each coarse value appears as a shifted diagonal stencil on the larger grid.

In [ ]:
plt.figure(figsize=(4, 3)) # create a grid-value heatmap.
plt.imshow(y_a3, cmap="magma") # visualize sparse painted output.
plt.colorbar(label="value") # add color scale.
plt.title("Advanced 3: scattered learned patches") # title plot.
plt.show() # display heatmap.

▶ What you'll see: zeros remain where no stencil weight wrote a value.

👀 Takeaway: 2-D transposed convolution is patch scattering plus summation on a larger canvas.

### Advanced 4 — Diagnose checkerboard coverage in 2-D

**Goal.** Count 2-D transposed-conv overlap, because uneven 1-D coverage multiplies across axes into checkerboard patterns. We build it in 4 steps.

In [ ]:
h_a4, w_a4 = 4, 4 # coarse grid size.
s_a4 = 2 # stride.
k_a4 = 3 # kernel size with uneven overlap for stride 2.
out_h_a4 = (h_a4 - 1) * s_a4 + k_a4 # output height.
out_w_a4 = (w_a4 - 1) * s_a4 + k_a4 # output width.
coverage_a4 = np.zeros((out_h_a4, out_w_a4)) # allocate 2-D coverage counts.
print("output shape:", coverage_a4.shape) # inspect expanded size.

▶ What you'll see: a 4×4 coarse grid expands to a 9×9 coverage map.

In [ ]:
for r_a4 in range(h_a4): # input rows.
    for c_a4 in range(w_a4): # input columns.
        coverage_a4[r_a4 * s_a4:r_a4 * s_a4 + k_a4, c_a4 * s_a4:c_a4 * s_a4 + k_a4] += 1 # count one all-one stencil.
print("unique coverage counts:", np.unique(coverage_a4)) # inspect overlap levels.
assert set(np.unique(coverage_a4)) == {1.0, 2.0, 4.0}

In [ ]:
plt.figure(figsize=(4, 3.5)) # create coverage heatmap.
plt.imshow(coverage_a4, cmap="magma") # visualize overlap counts.
plt.colorbar(label="contributors") # add count scale.
plt.title("Advanced 4: 2-D uneven overlap") # title plot.
plt.show() # display heatmap.

▶ What you'll see: a checkerboard-like map where some pixels have 4 contributors and others have 1 or 2.

In [ ]:
counts_a4, freqs_a4 = np.unique(coverage_a4, return_counts=True) # summarize coverage distribution.
print("coverage histogram:", dict(zip(counts_a4.astype(int), freqs_a4.astype(int)))) # inspect how many cells get each count.
plt.figure(figsize=(4, 2.8)) # create bar chart.
plt.bar(counts_a4.astype(str), freqs_a4, color="crimson") # plot histogram.
plt.title("Advanced 4: coverage-count distribution") # title plot.
plt.xlabel("contributors") # label contributors.
plt.ylabel("pixels") # label pixel count.
plt.show() # display chart.

▶ What you'll see: many output cells receive different numbers of contributions even for constant input.

👀 Takeaway: checkerboards come from deterministic overlap arithmetic before training even begins.

### Advanced 5 — Align an encoder and decoder shape

**Goal.** Choose transposed-conv settings that recover a target decoder length, because skip connections fail when shapes are off by one. We build it in 4 steps.

In [ ]:
target_a5 = 8 # desired high-resolution length for a skip connection.
n_in_a5 = 4 # coarse decoder feature length.
s_a5 = 2 # stride used to upsample.
k_a5 = 3 # transposed kernel size.
p_a5_options = np.array([0, 1, 2]) # candidate padding values.
o_a5_options = np.array([0, 1]) # candidate output padding values.
print("target length:", target_a5) # inspect target.

▶ What you'll see: the decoder must produce length 8.

In [ ]:
candidates_a5 = [] # store all shape candidates.
for p_cur_a5 in p_a5_options: # try each padding.
    for o_cur_a5 in o_a5_options: # try each output padding.
        length_cur_a5 = (n_in_a5 - 1) * s_a5 - 2 * p_cur_a5 + k_a5 + o_cur_a5 # formula.
        candidates_a5.append((p_cur_a5, o_cur_a5, length_cur_a5)) # store setting and output length.
print("candidates (p, o, length):", candidates_a5) # inspect all settings.

In [ ]:
matches_a5 = [(p_cur_a5, o_cur_a5) for p_cur_a5, o_cur_a5, length_cur_a5 in candidates_a5 if length_cur_a5 == target_a5] # find exact matches.
print("exact matches:", matches_a5) # inspect valid shape choices.
assert matches_a5 == [(1, 1)]

▶ What you'll see: padding 1 and output padding 1 are required for length 8.

In [ ]:
lengths_a5 = np.array([length_cur_a5 for _, _, length_cur_a5 in candidates_a5]) # collect lengths.
labels_a5 = [f"p={p_cur_a5},o={o_cur_a5}" for p_cur_a5, o_cur_a5, _ in candidates_a5] # make readable labels.
plt.figure(figsize=(6, 3)) # create shape-search plot.
plt.bar(labels_a5, lengths_a5, color="steelblue") # plot candidate lengths.
plt.axhline(target_a5, color="crimson", linestyle="--", label="target") # mark target length.
plt.title("Advanced 5: decoder shape alignment") # title plot.
plt.ylabel("output length") # label length axis.
plt.xticks(rotation=25) # rotate setting labels.
plt.legend() # show target label.
plt.show() # display plot.

▶ What you'll see: only one candidate bar lands exactly on the target line.

👀 Takeaway: decoder shape alignment is formula work; output padding fixes one-cell ambiguity when stride arithmetic needs it.